In [1]:
import subprocess
import datetime
import pandas as pd
import numpy as np
import joblib
from scapy.all import sniff
from collections import defaultdict
print("All libraries loaded!")

All libraries loaded!


In [2]:
# Load model and scaler from Week 4
model = joblib.load('threat_detection_model_3class.pkl')
scaler = joblib.load('scaler_3class.pkl')
top_features = pd.read_csv('top_features.csv')['0'].tolist()

# Label mapping
label_map = {0: 'BENIGN', 1: 'FTP-Patator', 2: 'SSH-Patator'}

print("Model loaded!")
print("Watching for these attacks:", [label_map[i] for i in [1,2]])

Model loaded!
Watching for these attacks: ['FTP-Patator', 'SSH-Patator']


In [3]:
# Load 15-class model instead
model = joblib.load('threat_detection_model_15class.pkl')
scaler = joblib.load('scaler_15class.pkl')
le = joblib.load('label_encoder_15class.pkl')
top_features = pd.read_csv('top_features_full.csv')['0'].tolist()

# Label mapping from encoder
label_map = {i: label for i, label in enumerate(le.classes_)}

print("15-class model loaded!")
print("\nCan detect these attacks:")
for i, label in label_map.items():
    if label != 'BENIGN':
        print(f"  {i}. {label}")

15-class model loaded!

Can detect these attacks:
  1. Bot
  2. DDoS
  3. DoS GoldenEye
  4. DoS Hulk
  5. DoS Slowhttptest
  6. DoS slowloris
  7. FTP-Patator
  8. Heartbleed
  9. Infiltration
  10. PortScan
  11. SSH-Patator
  12. Web Attack - Brute Force
  13. Web Attack - Sql Injection
  14. Web Attack - XSS


In [4]:
def capture_and_extract_features(duration=10):
    print(f"Capturing traffic for {duration} seconds...")
    packets = sniff(timeout=duration)
    flows = defaultdict(list)
    
    for pkt in packets:
        if pkt.haslayer('IP') and pkt.haslayer('TCP'):
            src = pkt['IP'].src
            dst = pkt['IP'].dst
            sport = pkt['TCP'].sport
            dport = pkt['TCP'].dport
            flow_key = tuple(sorted([f"{src}:{sport}", f"{dst}:{dport}"]))
            flows[flow_key].append({
                'time': pkt.time,
                'length': len(pkt),
                'flags': str(pkt['TCP'].flags),
                'src': src,
                'dst': dst,
                'sport': sport,
                'dport': dport
            })
    
    print(f"Captured {len(packets)} packets across {len(flows)} flows")
    return flows

def calculate_flow_features_complete(flows):
    flow_features = []
    
    for flow_key, packets in flows.items():
        if len(packets) < 2:
            continue
        
        packets_sorted = sorted(packets, key=lambda x: x['time'])
        duration = (packets_sorted[-1]['time'] - packets_sorted[0]['time']) * 1_000_000
        duration = duration if duration > 0 else 1
        
        forward_src = packets_sorted[0]['src']
        fwd_packets = [p for p in packets_sorted if p['src'] == forward_src]
        bwd_packets = [p for p in packets_sorted if p['src'] != forward_src]
        
        fwd_lengths = [p['length'] for p in fwd_packets] or [0]
        bwd_lengths = [p['length'] for p in bwd_packets] or [0]
        all_lengths = [p['length'] for p in packets_sorted]
        
        times = [p['time'] for p in packets_sorted]
        iat = [times[i+1]-times[i] for i in range(len(times)-1)] or [0]
        
        syn_count = sum(1 for p in packets_sorted if 'S' in p['flags'])
        psh_count_fwd = sum(1 for p in fwd_packets if 'P' in p['flags'])
        
        # Get the actual source IP of the attacker (forward direction source)
        attacker_ip = packets_sorted[0]['src']
        dest_port = packets_sorted[0]['dport']
        
        features = {
            'attacker_ip': attacker_ip,
            'Total_Length_of_Bwd_Packets': sum(bwd_lengths),
            'Packet_Length_Variance': np.var(all_lengths),
            'Fwd_Packet_Length_Max': max(fwd_lengths),
            'Subflow_Fwd_Bytes': sum(fwd_lengths),
            'Packet_Length_Std': np.std(all_lengths),
            'Bwd_Packet_Length_Mean': np.mean(bwd_lengths),
            'Max_Packet_Length': max(all_lengths),
            'Subflow_Bwd_Bytes': sum(bwd_lengths),
            'Average_Packet_Size': np.mean(all_lengths),
            'Destination_Port': dest_port,
            'Init_Win_bytes_forward': fwd_lengths[0] if fwd_lengths else 0,
            'Avg_Bwd_Segment_Size': np.mean(bwd_lengths),
            'Packet_Length_Mean': np.mean(all_lengths),
            'Total_Length_Fwd_Packets': sum(fwd_lengths),
            'Bwd_Packet_Length_Std': np.std(bwd_lengths),
            'PSH_Flag_Count': psh_count_fwd,
            'Total_Backward_Packets': len(bwd_packets),
            'Subflow_Fwd_Packets': len(fwd_packets),
            'Fwd_Header_Length': len(fwd_packets) * 20,
            'Fwd_Header_Length_1': len(fwd_packets) * 20,
        }
        
        flow_features.append(features)
    
    return pd.DataFrame(flow_features)

print("Feature extraction functions ready!")

Feature extraction functions ready!


In [5]:
# Column mapping for 15-class model features
column_mapping_15 = {
    'Total_Length_of_Bwd_Packets': ' Total Length of Bwd Packets',
    'Packet_Length_Variance': ' Packet Length Variance',
    'Fwd_Packet_Length_Max': ' Fwd Packet Length Max',
    'Subflow_Fwd_Bytes': ' Subflow Fwd Bytes',
    'Packet_Length_Std': ' Packet Length Std',
    'Bwd_Packet_Length_Mean': ' Bwd Packet Length Mean',
    'Max_Packet_Length': ' Max Packet Length',
    'Subflow_Bwd_Bytes': ' Subflow Bwd Bytes',
    'Average_Packet_Size': ' Average Packet Size',
    'Destination_Port': ' Destination Port',
    'Init_Win_bytes_forward': 'Init_Win_bytes_forward',
    'Avg_Bwd_Segment_Size': ' Avg Bwd Segment Size',
    'Packet_Length_Mean': ' Packet Length Mean',
    'Total_Length_Fwd_Packets': 'Total Length of Fwd Packets',
    'Bwd_Packet_Length_Std': ' Bwd Packet Length Std',
    'PSH_Flag_Count': ' PSH Flag Count',
    'Total_Backward_Packets': ' Total Backward Packets',
    'Subflow_Fwd_Packets': 'Subflow Fwd Packets',
    'Fwd_Header_Length': ' Fwd Header Length',
    'Fwd_Header_Length_1': ' Fwd Header Length.1',
}

print("Column mapping ready!")

Column mapping ready!


In [6]:
def intrusion_prevention_system(duration=10, auto_block=True):
    print("="*60)
    print("🛡️  INTRUSION PREVENTION SYSTEM — ACTIVE")
    print("="*60)
    
    # Step 1: Capture live traffic
    flows = capture_and_extract_features(duration=duration)
    df_feat = calculate_flow_features_complete(flows)
    
    if len(df_feat) == 0:
        print("No flows captured.")
        return
    
    # Step 2: Extract attacker IPs before dropping that column
    attacker_ips = df_feat['attacker_ip'].values
    df_feat = df_feat.drop('attacker_ip', axis=1)
    
    # Step 3: Align columns to model's expected format
    df_renamed = df_feat.rename(columns=column_mapping_15)
    df_final = df_renamed[top_features]
    
    # Step 4: Scale and predict
    X_scaled = scaler.transform(df_final)
    predictions = model.predict(X_scaled)
    probas = model.predict_proba(X_scaled)
    
    # Step 5: Alert and block
    print(f"\n{'='*60}")
    print("THREAT ANALYSIS REPORT")
    print(f"{'='*60}")
    
    threats_detected = 0
    threats_blocked = 0
    
    for i, (pred, proba, attacker_ip) in enumerate(zip(predictions, probas, attacker_ips)):
        label = label_map[pred]
        confidence = max(proba) * 100
        port = df_final.iloc[i][' Destination Port']
        
        if label == 'BENIGN':
            print(f"✅ Flow {i}: Normal traffic on port {int(port)} (Confidence: {confidence:.1f}%)")
        else:
            threats_detected += 1
            print(f"\n🚨 THREAT DETECTED!")
            print(f"   Attack Type : {label}")
            print(f"   Source IP   : {attacker_ip}")
            print(f"   Target Port : {int(port)}")
            print(f"   Confidence  : {confidence:.1f}%")
            
            if auto_block:
                blocked = block_ip(attacker_ip, f"{label} attack detected on port {int(port)}")
                if blocked:
                    threats_blocked += 1
    
    # Step 6: Summary
    print(f"\n{'='*60}")
    print(f"SUMMARY")
    print(f"{'='*60}")
    print(f"Total flows analyzed : {len(df_final)}")
    print(f"Threats detected     : {threats_detected}")
    print(f"IPs blocked          : {threats_blocked}")
    print(f"IPs already blocked  : {len(blocked_ips)}")
    
    # Step 7: Save block log
    if block_log:
        pd.DataFrame(block_log).to_csv('block_log.csv', index=False)
        print(f"📝 Block log saved to block_log.csv")
    
    print("="*60)

print("Intrusion Prevention System ready!")

Intrusion Prevention System ready!


In [7]:
intrusion_prevention_system(duration=10, auto_block=True)


🛡️  INTRUSION PREVENTION SYSTEM — ACTIVE
Capturing traffic for 10 seconds...
Captured 392 packets across 10 flows

THREAT ANALYSIS REPORT
✅ Flow 0: Normal traffic on port 56877 (Confidence: 100.0%)
✅ Flow 1: Normal traffic on port 443 (Confidence: 96.7%)
✅ Flow 2: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 3: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 4: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 5: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 6: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 7: Normal traffic on port 443 (Confidence: 96.7%)
✅ Flow 8: Normal traffic on port 60130 (Confidence: 93.3%)
✅ Flow 9: Normal traffic on port 443 (Confidence: 100.0%)

SUMMARY
Total flows analyzed : 10
Threats detected     : 0
IPs blocked          : 0


NameError: name 'blocked_ips' is not defined

In [8]:
# Initialize tracking variables
blocked_ips = set()
block_log = []

def block_ip(ip_address, reason):
    if ip_address in blocked_ips:
        print(f"⚠️ {ip_address} already blocked — skipping")
        return False
    try:
        rule_name = f"CyberSecBlock_{ip_address.replace('.', '_')}"
        command = [
            'netsh', 'advfirewall', 'firewall', 'add', 'rule',
            f'name={rule_name}',
            'dir=in',
            'action=block',
            f'remoteip={ip_address}',
            'enable=yes'
        ]
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            blocked_ips.add(ip_address)
            block_log.append({
                'Timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'IP_Blocked': ip_address,
                'Reason': reason,
                'Action': 'BLOCKED',
                'Status': 'SUCCESS'
            })
            print(f"🚫 BLOCKED: {ip_address} — Reason: {reason}")
            return True
        else:
            print(f"❌ Failed to block {ip_address}: {result.stderr}")
            return False
    except Exception as e:
        print(f"❌ Error blocking {ip_address}: {e}")
        return False

def unblock_ip(ip_address):
    rule_name = f"CyberSecBlock_{ip_address.replace('.', '_')}"
    command = [
        'netsh', 'advfirewall', 'firewall', 'delete', 'rule',
        f'name={rule_name}'
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode == 0:
        blocked_ips.discard(ip_address)
        print(f"✅ UNBLOCKED: {ip_address}")
    else:
        print(f"❌ Failed to unblock {ip_address}: {result.stderr}")

def intrusion_prevention_system(duration=10, auto_block=True):
    print("="*60)
    print("🛡️  INTRUSION PREVENTION SYSTEM — ACTIVE")
    print("="*60)
    
    flows = capture_and_extract_features(duration=duration)
    df_feat = calculate_flow_features_complete(flows)
    
    if len(df_feat) == 0:
        print("No flows captured.")
        return
    
    attacker_ips = df_feat['attacker_ip'].values
    df_feat = df_feat.drop('attacker_ip', axis=1)
    df_renamed = df_feat.rename(columns=column_mapping_15)
    df_final = df_renamed[top_features]
    
    X_scaled = scaler.transform(df_final)
    predictions = model.predict(X_scaled)
    probas = model.predict_proba(X_scaled)
    
    print(f"\n{'='*60}")
    print("THREAT ANALYSIS REPORT")
    print(f"{'='*60}")
    
    threats_detected = 0
    threats_blocked = 0
    
    for i, (pred, proba, attacker_ip) in enumerate(zip(predictions, probas, attacker_ips)):
        label = label_map[pred]
        confidence = max(proba) * 100
        port = df_final.iloc[i][' Destination Port']
        
        if label == 'BENIGN':
            print(f"✅ Flow {i}: Normal traffic on port {int(port)} (Confidence: {confidence:.1f}%)")
        else:
            threats_detected += 1
            print(f"\n🚨 THREAT DETECTED!")
            print(f"   Attack Type : {label}")
            print(f"   Source IP   : {attacker_ip}")
            print(f"   Target Port : {int(port)}")
            print(f"   Confidence  : {confidence:.1f}%")
            if auto_block:
                blocked = block_ip(attacker_ip, f"{label} attack detected on port {int(port)}")
                if blocked:
                    threats_blocked += 1
    
    print(f"\n{'='*60}")
    print(f"SUMMARY")
    print(f"{'='*60}")
    print(f"Total flows analyzed : {len(df_final)}")
    print(f"Threats detected     : {threats_detected}")
    print(f"IPs blocked          : {threats_blocked}")
    print(f"IPs already blocked  : {len(blocked_ips)}")
    
    if block_log:
        pd.DataFrame(block_log).to_csv('block_log.csv', index=False)
        print(f"📝 Block log saved to block_log.csv")
    
    print("="*60)

print("IPS fully ready!")

IPS fully ready!


In [9]:
intrusion_prevention_system(duration=10, auto_block=True)

🛡️  INTRUSION PREVENTION SYSTEM — ACTIVE
Capturing traffic for 10 seconds...
Captured 393 packets across 15 flows

THREAT ANALYSIS REPORT
✅ Flow 0: Normal traffic on port 58592 (Confidence: 87.9%)
✅ Flow 1: Normal traffic on port 58593 (Confidence: 83.4%)
✅ Flow 2: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 3: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 4: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 5: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 6: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 7: Normal traffic on port 58605 (Confidence: 96.7%)
✅ Flow 8: Normal traffic on port 443 (Confidence: 96.7%)
✅ Flow 9: Normal traffic on port 443 (Confidence: 96.7%)
✅ Flow 10: Normal traffic on port 443 (Confidence: 96.7%)
✅ Flow 11: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 12: Normal traffic on port 443 (Confidence: 96.7%)
✅ Flow 13: Normal traffic on port 443 (Confidence: 100.0%)
✅ Flow 14: Normal traffic on port 443 (Confiden

In [10]:
# Simulate a detected attack and test blocking
print("Testing IP blocking mechanism...")
print()

# Simulate as if our IPS detected an SSH attack from this IP
test_ip = "192.168.1.100"  # fake attacker IP for testing

block_ip(test_ip, "SSH-Patator attack detected on port 22 (TEST)")

print()
print("Currently blocked IPs:", blocked_ips)
print()

# Now test that it handles duplicate blocking correctly
print("Trying to block same IP again:")
block_ip(test_ip, "Second attempt (TEST)")

print()
print("Now unblocking test IP...")
unblock_ip(test_ip)
print("Blocked IPs after unblock:", blocked_ips)

Testing IP blocking mechanism...

🚫 BLOCKED: 192.168.1.100 — Reason: SSH-Patator attack detected on port 22 (TEST)

Currently blocked IPs: {'192.168.1.100'}

Trying to block same IP again:
⚠️ 192.168.1.100 already blocked — skipping

Now unblocking test IP...
✅ UNBLOCKED: 192.168.1.100
Blocked IPs after unblock: set()


In [11]:
# Rule-based threat response system
THREAT_RULES = {
    'port_22': {
        'description': 'SSH Brute Force Attempt',
        'condition': lambda port, flags, packets_per_s: port == 22 and packets_per_s > 100,
        'action': 'BLOCK',
        'severity': 'HIGH'
    },
    'port_21': {
        'description': 'FTP Brute Force Attempt',
        'condition': lambda port, flags, packets_per_s: port == 21 and packets_per_s > 100,
        'action': 'BLOCK',
        'severity': 'HIGH'
    },
    'high_volume': {
        'description': 'Possible DDoS Attack',
        'condition': lambda port, flags, packets_per_s: packets_per_s > 10000,
        'action': 'BLOCK',
        'severity': 'CRITICAL'
    },
    'syn_flood': {
        'description': 'SYN Flood Attack',
        'condition': lambda port, flags, packets_per_s: 'S' in flags and packets_per_s > 5000,
        'action': 'BLOCK',
        'severity': 'CRITICAL'
    },
}

def apply_rules(port, flags, packets_per_s, attacker_ip):
    """Check traffic against rule-based system"""
    for rule_name, rule in THREAT_RULES.items():
        try:
            if rule['condition'](port, flags, packets_per_s):
                print(f"⚠️  RULE TRIGGERED: {rule['description']}")
                print(f"   Severity : {rule['severity']}")
                print(f"   Action   : {rule['action']}")
                if rule['action'] == 'BLOCK':
                    block_ip(attacker_ip, rule['description'])
                return rule_name
        except:
            pass
    return None

print("Rule-based system ready!")
print(f"Total rules loaded: {len(THREAT_RULES)}")
for name, rule in THREAT_RULES.items():
    print(f"  - {rule['description']} [{rule['severity']}]")

Rule-based system ready!
Total rules loaded: 4
  - SSH Brute Force Attempt [HIGH]
  - FTP Brute Force Attempt [HIGH]
  - Possible DDoS Attack [CRITICAL]
  - SYN Flood Attack [CRITICAL]


In [12]:
def combined_ips(duration=10, auto_block=True):
    print("="*60)
    print("🛡️  COMBINED AI + RULE-BASED IPS — ACTIVE")
    print("="*60)
    
    flows = capture_and_extract_features(duration=duration)
    df_feat = calculate_flow_features_complete(flows)
    
    if len(df_feat) == 0:
        print("No flows captured.")
        return
    
    attacker_ips = df_feat['attacker_ip'].values
    raw_features = df_feat.copy()
    
    df_feat = df_feat.drop('attacker_ip', axis=1)
    df_renamed = df_feat.rename(columns=column_mapping_15)
    df_final = df_renamed[top_features]
    
    X_scaled = scaler.transform(df_final)
    predictions = model.predict(X_scaled)
    probas = model.predict_proba(X_scaled)
    
    print(f"\n{'='*60}")
    print("COMBINED THREAT ANALYSIS")
    print(f"{'='*60}")
    
    threats_detected = 0
    rule_triggers = 0
    
    for i, (pred, proba, attacker_ip) in enumerate(zip(predictions, probas, attacker_ips)):
        label = label_map[pred]
        confidence = max(proba) * 100
        port = int(df_final.iloc[i][' Destination Port'])
        packets_per_s = raw_features.iloc[i]['Total_Length_of_Bwd_Packets']
        flags = raw_features.iloc[i].get('flags', '')
        
        print(f"\n--- Flow {i} | Port {port} | Source: {attacker_ip} ---")
        
        # AI Detection
        if label != 'BENIGN':
            threats_detected += 1
            print(f"🤖 AI DETECTED: {label} (Confidence: {confidence:.1f}%)")
            if auto_block:
                block_ip(attacker_ip, f"AI: {label} on port {port}")
        else:
            print(f"🤖 AI: Normal traffic (Confidence: {confidence:.1f}%)")
        
        # Rule Based Detection
        rule_triggered = apply_rules(port, flags, packets_per_s, attacker_ip)
        if rule_triggered:
            rule_triggers += 1
    
    print(f"\n{'='*60}")
    print("FINAL SUMMARY")
    print(f"{'='*60}")
    print(f"Flows analyzed    : {len(df_final)}")
    print(f"AI threats found  : {threats_detected}")
    print(f"Rules triggered   : {rule_triggers}")
    print(f"Total IPs blocked : {len(blocked_ips)}")
    
    if block_log:
        pd.DataFrame(block_log).to_csv('block_log.csv', index=False)
        print(f"📝 Block log saved!")
    
    print("="*60)

# Run it!
combined_ips(duration=10, auto_block=True)

🛡️  COMBINED AI + RULE-BASED IPS — ACTIVE
Capturing traffic for 10 seconds...
Captured 227 packets across 11 flows

COMBINED THREAT ANALYSIS

--- Flow 0 | Port 56877 | Source: 20.184.175.5 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 1 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 96.7%)

--- Flow 2 | Port 54775 | Source: 32.192.94.125 ---
🤖 AI: Normal traffic (Confidence: 93.3%)

--- Flow 3 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 4 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)
⚠️  RULE TRIGGERED: Possible DDoS Attack
   Severity : CRITICAL
   Action   : BLOCK
🚫 BLOCKED: 192.168.120.41 — Reason: Possible DDoS Attack

--- Flow 5 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 96.7%)

--- Flow 6 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 7 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Norm

In [13]:
unblock_ip("192.168.120.41")

✅ UNBLOCKED: 192.168.120.41


In [14]:
# Fixed and improved rules
THREAT_RULES = {
    'ssh_brute_force': {
        'description': 'SSH Brute Force Attempt',
        'condition': lambda port, fwd_packets, duration: port == 22 and fwd_packets > 20 and duration < 60,
        'action': 'BLOCK',
        'severity': 'HIGH'
    },
    'ftp_brute_force': {
        'description': 'FTP Brute Force Attempt',
        'condition': lambda port, fwd_packets, duration: port == 21 and fwd_packets > 20 and duration < 60,
        'action': 'BLOCK',
        'severity': 'HIGH'
    },
    'port_scan': {
        'description': 'Port Scanning Detected',
        'condition': lambda port, fwd_packets, duration: fwd_packets == 1 and duration < 1,
        'action': 'ALERT',
        'severity': 'MEDIUM'
    },
    'syn_flood': {
        'description': 'SYN Flood Attack',
        'condition': lambda port, fwd_packets, duration: fwd_packets > 1000 and duration < 10,
        'action': 'BLOCK',
        'severity': 'CRITICAL'
    },
}

def apply_rules_fixed(port, fwd_packets, duration_seconds, attacker_ip):
    for rule_name, rule in THREAT_RULES.items():
        try:
            if rule['condition'](port, fwd_packets, duration_seconds):
                print(f"⚠️  RULE TRIGGERED: {rule['description']}")
                print(f"   Severity : {rule['severity']}")
                print(f"   Action   : {rule['action']}")
                if rule['action'] == 'BLOCK':
                    block_ip(attacker_ip, rule['description'])
                return rule_name
        except:
            pass
    return None

print("Fixed rules loaded!")
for name, rule in THREAT_RULES.items():
    print(f"  - {rule['description']} [{rule['severity']}] → {rule['action']}")

Fixed rules loaded!
  - SSH Brute Force Attempt [HIGH] → BLOCK
  - FTP Brute Force Attempt [HIGH] → BLOCK
  - Port Scanning Detected [MEDIUM] → ALERT
  - SYN Flood Attack [CRITICAL] → BLOCK


In [15]:
def combined_ips_fixed(duration=10, auto_block=True):
    print("="*60)
    print("🛡️  COMBINED AI + RULE-BASED IPS v2.0 — ACTIVE")
    print("="*60)
    
    flows = capture_and_extract_features(duration=duration)
    df_feat = calculate_flow_features_complete(flows)
    
    if len(df_feat) == 0:
        print("No flows captured.")
        return
    
    attacker_ips = df_feat['attacker_ip'].values
    raw_features = df_feat.copy()
    
    df_feat = df_feat.drop('attacker_ip', axis=1)
    df_renamed = df_feat.rename(columns=column_mapping_15)
    df_final = df_renamed[top_features]
    
    X_scaled = scaler.transform(df_final)
    predictions = model.predict(X_scaled)
    probas = model.predict_proba(X_scaled)
    
    print(f"\n{'='*60}")
    print("COMBINED THREAT ANALYSIS")
    print(f"{'='*60}")
    
    threats_detected = 0
    rule_triggers = 0
    
    for i, (pred, proba, attacker_ip) in enumerate(zip(predictions, probas, attacker_ips)):
        label = label_map[pred]
        confidence = max(proba) * 100
        port = int(df_final.iloc[i][' Destination Port'])
        fwd_packets = int(raw_features.iloc[i]['Subflow_Fwd_Packets'])
        duration_s = raw_features.iloc[i]['Total_Length_of_Bwd_Packets'] / 1_000_000
        
        print(f"\n--- Flow {i} | Port {port} | Source: {attacker_ip} ---")
        
        # AI Detection
        if label != 'BENIGN':
            threats_detected += 1
            print(f"🤖 AI DETECTED: {label} (Confidence: {confidence:.1f}%)")
            if auto_block:
                block_ip(attacker_ip, f"AI: {label} on port {port}")
        else:
            print(f"🤖 AI: Normal traffic (Confidence: {confidence:.1f}%)")
        
        # Rule Based Detection
        rule_triggered = apply_rules_fixed(port, fwd_packets, duration_s, attacker_ip)
        if rule_triggered:
            rule_triggers += 1
    
    print(f"\n{'='*60}")
    print("FINAL SUMMARY")
    print(f"{'='*60}")
    print(f"Flows analyzed    : {len(df_final)}")
    print(f"AI threats found  : {threats_detected}")
    print(f"Rules triggered   : {rule_triggers}")
    print(f"Total IPs blocked : {len(blocked_ips)}")
    
    if block_log:
        pd.DataFrame(block_log).to_csv('block_log.csv', index=False)
        print(f"📝 Block log saved!")
    
    print("="*60)

# Run it!
combined_ips_fixed(duration=10, auto_block=True)

🛡️  COMBINED AI + RULE-BASED IPS v2.0 — ACTIVE
Capturing traffic for 10 seconds...
Captured 1244 packets across 19 flows

COMBINED THREAT ANALYSIS

--- Flow 0 | Port 57538 | Source: 3.220.207.191 ---
🤖 AI: Normal traffic (Confidence: 93.3%)

--- Flow 1 | Port 60267 | Source: 100.57.236.166 ---
🤖 AI: Normal traffic (Confidence: 90.0%)

--- Flow 2 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 3 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 4 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 5 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 6 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 7 | Port 57531 | Source: 54.160.252.172 ---
🤖 AI: Normal traffic (Confidence: 100.0%)

--- Flow 8 | Port 443 | Source: 192.168.120.41 ---
🤖 AI: Normal traffic (Confidence: 96.7%)

--- 